In [ ]:
!pip install easyocr
!pip install opencv-python
!pip install transformers
!pip install torchvision
!pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.8/422.8 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import re
import pandas as pd
import numpy as np
import torch
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import easyocr
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer, BertModel

In [ ]:
IMAGE_DIR = "/content/drive/MyDrive/CIS515/bookcovers_asu/"
ASU_CSV = "/content/drive/MyDrive/CIS515/bookcover_asu.csv"
ASU_SAVE = "/content/drive/MyDrive/CIS515/bookcover_asu_with_ocr.csv"

DATA_CSV = "/content/drive/MyDrive/CIS515/bookcover_data.csv"
DATA_SAVE = "/content/drive/MyDrive/CIS515/bookcover_data_with_ocr.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OCR Module (EasyOCR)

In [ ]:
def init_ocr():
    return easyocr.Reader(['en'])

def extract_text_from_image(reader, image_path):
    try:
        result = reader.readtext(image_path, detail=0)
        clean = ' '.join([re.sub(r'[^a-zA-Z0-9\s]', '', text) for text in result])
        return clean.strip()
    except Exception as e:
        print(f"OCR failed for {image_path}: {e}")
        return ""

Feature Extraction (ResNet + BERT)

In [ ]:
def init_resnet():
    model = models.resnet50(pretrained=True)
    model = torch.nn.Sequential(*list(model.children())[:-1])
    return model.to(DEVICE).eval()

IMG_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

def extract_image_feature(model, image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        tensor = IMG_TRANSFORM(image).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            feature = model(tensor)
        return feature.cpu().numpy().flatten()
    except Exception as e:
        print(f"Image error: {image_path} - {e}")
        return np.zeros(2048)

def init_bert():
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased').to(DEVICE).eval()
    return tokenizer, model

def extract_text_feature(text, tokenizer, model):
    try:
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            output = model(**inputs)
        return output.last_hidden_state.mean(dim=1).cpu().numpy().flatten()
    except:
        return np.zeros(768)

Dataset Processing Function

In [ ]:
def process_dataset(csv_path, save_path, image_dir, filename_col,
                    ocr_reader, resnet_model, tokenizer, bert_model):
    print(f"\nProcessing: {os.path.basename(csv_path)}")

    df = pd.read_csv(csv_path)

    # Ensure image filenames have .jpg extension
    df["__Filename"] = df[filename_col].astype(str).apply(lambda x: x if x.endswith(".jpg") else x + ".jpg")

    # Filter missing images
    df = df[df["__Filename"].apply(lambda x: os.path.exists(os.path.join(image_dir, x)))].reset_index(drop=True)

    # OCR
    print("Running OCR...")
    df["OCR_Text"] = [
        extract_text_from_image(ocr_reader, os.path.join(image_dir, fname))
        for fname in tqdm(df["__Filename"])
    ]

    # Image Features
    print("Extracting image features...")
    image_features = np.array([
        extract_image_feature(resnet_model, os.path.join(image_dir, fname))
        for fname in tqdm(df["__Filename"])
    ])

    # Text Features
    print("Extracting text features...")
    text_features = np.array([
        extract_text_feature(text, tokenizer, bert_model)
        for text in tqdm(df["OCR_Text"])
    ])

    # Save processed CSV
    df.drop(columns=["__Filename"]).to_csv(save_path, index=False)
    print(f"Saved OCR-processed data to: {save_path}")

    return df, image_features, text_features

Matching Function

In [ ]:
def match_book(query_image_path, reader, resnet_model, tokenizer, bert_model,
               image_features_matrix, text_features_matrix, reference_df):

    query_text = extract_text_from_image(reader, query_image_path)
    query_img_feat = extract_image_feature(resnet_model, query_image_path)
    query_text_feat = extract_text_feature(query_text, tokenizer, bert_model)

    sim_img = cosine_similarity([query_img_feat], image_features_matrix)[0]
    sim_txt = cosine_similarity([query_text_feat], text_features_matrix)[0]
    combined = 0.5 * sim_img + 0.5 * sim_txt

    top_idx = np.argmax(combined)
    matched = reference_df.iloc[top_idx]

    return {
        "Query OCR Text": query_text,
        "Matched Title": matched["Title"],
        "Matched Author": matched["Author"],
        "Similarity Score": combined[top_idx]
    }

Run the Full Pipeline

In [ ]:
# Step 1: Initialize models
ocr_reader = init_ocr()
resnet_model = init_resnet()
tokenizer, bert_model = init_bert()

# Step 2: Load ASU OCR data (already processed)
asu_df = pd.read_csv(ASU_SAVE)  # "/"

# Step 3: Extract image & text features again (or load from saved npy if you have)
asu_img_feat = np.array([
    extract_image_feature(resnet_model, os.path.join(IMAGE_DIR, fname if fname.endswith(".jpg") else fname + ".jpg"))
    for fname in tqdm(asu_df["Filename"])
])

asu_txt_feat = np.array([
    extract_text_feature(text, tokenizer, bert_model)
    for text in tqdm(asu_df["OCR_Text"])
])

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 148MB/s]
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, 

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/CIS515/bookcover_asu_with_ocr.csv'

In [ ]:
np.save("/content/drive/MyDrive/CIS515/asu_img_feat.npy", asu_img_feat)
np.save("/content/drive/MyDrive/CIS515/asu_txt_feat.npy", asu_txt_feat)

NameError: name 'asu_img_feat' is not defined

In [ ]:
def match_book_with_database(query_image_path, reader, resnet_model, tokenizer, bert_model,
                              image_features_matrix, text_features_matrix, reference_df):
    query_text = extract_text_from_image(reader, query_image_path)
    query_img_feat = extract_image_feature(resnet_model, query_image_path)
    query_text_feat = extract_text_feature(query_text, tokenizer, bert_model)

    sim_img = cosine_similarity([query_img_feat], image_features_matrix)[0]
    sim_txt = cosine_similarity([query_text_feat], text_features_matrix)[0]
    combined = 0.5 * sim_img + 0.5 * sim_txt

    top_idx = np.argmax(combined)
    matched_row = reference_df.iloc[top_idx]

    matched_title = matched_row["Title"].strip().lower()
    matched_author = matched_row["Author"].strip().lower()

    matched_in_db = reference_df[
        (reference_df["Title"].str.strip().str.lower() == matched_title) &
        (reference_df["Author"].str.strip().str.lower() == matched_author)
    ]

    if matched_in_db.empty:
        return {
            "status": "Not available at ASU",
            "ocr_text": query_text
        }

    match_info = matched_in_db.iloc[0]
    availability = match_info.get("Availability", "Unknown")
    category = match_info.get("Category", "Unknown")
    location = match_info.get("Library Location", "Unknown")

    if availability.strip().lower() == "available":
        return {
            "status": "Available",
            "ocr_text": query_text,
            "title": match_info["Title"],
            "author": match_info["Author"],
            "category": category,
            "availability": availability,
            "location": location
        }

    # Find alternative books in same category
    alternatives = reference_df[
        (reference_df["Category"] == category) &
        (reference_df["Availability"].str.lower() == "available") &
        ~(
            (reference_df["Title"].str.strip().str.lower() == matched_title) &
            (reference_df["Author"].str.strip().str.lower() == matched_author)
        )
    ]

    recommendations = alternatives[["Title", "Author", "Library Location"]].head(3).to_dict(orient="records")

    return {
        "status": f"{availability} (original book unavailable)",
        "ocr_text": query_text,
        "title": match_info["Title"],
        "author": match_info["Author"],
        "category": category,
        "recommendations": recommendations
    }

Book Matching and Availability Lookup


In [ ]:
query_img = os.path.join(IMAGE_DIR, "044654034X.jpg")  # change to your test file

result = match_book_with_database(
    query_image_path=query_img,
    reader=ocr_reader,
    resnet_model=resnet_model,
    tokenizer=tokenizer,
    bert_model=bert_model,
    image_features_matrix=asu_img_feat,
    text_features_matrix=asu_txt_feat,
    reference_df=asu_df
)

if result["status"] == "Available":
    print("Match Found:")
    print("Title:", result["title"])
    print("Author:", result["author"])
    print("Category:", result["category"])
    print("Availability:", result["availability"])
    print("Library Location:", result["location"])

elif result["status"] == "Not available at ASU":
    print("No match found in ASU database.")

else:
    print("Original book is not available.")
    print("Status:", result["status"])
    print("Matched Title:", result["title"])
    print("Matched Author:", result["author"])
    print("Category:", result["category"])
    print("Alternative available books:")
    for rec in result["recommendations"]:
        print(f"- {rec['Title']} by {rec['Author']} at {rec['Library Location']}")

Match Found:
Title: Big Book of Spelling Tests
Author: Orin Hargraves
Category: Reference
Availability: Available
Library Location: Noble Library
